In [35]:
!pip install bayesian-torch
!pip install curvlinops-for-pytorch==2.0
!pip install laplace-torch
!pip install torchmetrics
!pip install matplotlib
!pip install pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader,Dataset
from laplace import Laplace
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("Library Versions:")
print('numpy:',np.__version__)
print('pandas:',pd.__version__)
print('torch:',torch.__version__)


Library Versions:
numpy: 2.5.2
pandas: 3.0.5
torch: 2.13.0+cpu


In [37]:

n_epochs = 200
verbose_option = True

# Regression for Naval Plant Maintenance

Load dataset

In [38]:
class DummyDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [39]:
npm = pd.read_csv('navalplantmaintenance.csv',header=None)
npm_train, npm_test = train_test_split(npm,test_size=0.25,random_state=42)
npm_train_np = npm_train.to_numpy()
npm_test_np = npm_test.to_numpy()
x_train_np = npm_train_np[:,:16]
x_test_np = npm_test_np[:,:16]
y_train_np = npm_train_np[:,17]
y_test_np = npm_test_np[:,17]
x_mu = x_train_np.mean(axis=0)
x_sigma = x_train_np.std(axis=0)
y_mu=y_train_np.mean(axis=0)
y_sigma=y_train_np.std(axis=0)
def scale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return (x-x_mu)/x_sigma
def unscale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return x_sigma*x+x_mu
x_train_np_z = scale(x_train_np,x_mu,x_sigma)
y_train_np_z = scale(y_train_np,y_mu,y_sigma)
x_test_np_z = scale(x_test_np,x_mu,x_sigma)
y_test_np_z = scale(y_test_np,y_mu,y_sigma)
x_train_t_z = torch.FloatTensor(x_train_np_z)
y_train_t_z = torch.FloatTensor(y_train_np_z)
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)

In [40]:
train_t_z_dataset = DummyDataset(x_train_t_z,y_train_t_z)
train_t_z_dataloader = DataLoader(train_t_z_dataset)

1. Using PyTorch, perform variational inference using MC Dropout for a non-linear Gaussian
prediction model with heteroscedastic uncertainty for the regression dataset.

In [41]:
def nlls(y, mu, std):
    return torch.square(y - mu)/(2.0*torch.square(std))+torch.log(std)
n_train_examples = x_train_np_z.shape[0]

class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = torch.nn.Linear(inputSize, hiddenSize)
        self.layer2 = torch.nn.Linear(hiddenSize, hiddenSize)
        self.linear_mu = torch.nn.Linear(hiddenSize, outputSize)
        self.linear_sigma = torch.nn.Linear(hiddenSize, outputSize)

    def forward(self, x):
        h1 = torch.nn.functional.relu(self.layer1(torch.nn.functional.dropout(x, p=0.8, training=True))) # Run the first layer using dropout with p_drop = 0.8 and ReLU
        h2 = torch.nn.functional.relu(self.layer2(torch.nn.functional.dropout(h1, p=0.8, training=True))) # Run the second layer using dropout with p_drop = 0.8 and ReLU
        mu = self.linear_mu(h2)
        sigma = torch.nn.functional.softplus(self.linear_sigma(h2))
        return mu, sigma

    def L2reg(self):
        l2reg_sum = 0.0
        l2reg_sum += torch.square(self.layer1.weight).sum()
        l2reg_sum += torch.square(self.layer1.bias).sum()
        l2reg_sum += torch.square(self.layer2.weight).sum()
        l2reg_sum += torch.square(self.layer2.bias).sum()
        l2reg_sum += torch.square(self.linear_mu.weight).sum()
        l2reg_sum += torch.square(self.linear_mu.bias).sum()
        l2reg_sum += torch.square(self.linear_sigma.weight).sum()
        l2reg_sum += torch.square(self.linear_sigma.bias).sum()
        return l2reg_sum

In [42]:
model = nn(x_train_np_z.shape[1],50, 1)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)
l2_coeff = 1e-4
for i in range(50):
    mu,s= model(x_train_t_z)
    nll_loss = nlls(y_train_t_z, mu, s).mean()
    loss = nll_loss + l2_coeff * model.L2reg()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if verbose_option: print(i, loss)

0 tensor(0.7102, grad_fn=<AddBackward0>)
1 tensor(0.6407, grad_fn=<AddBackward0>)
2 tensor(0.6410, grad_fn=<AddBackward0>)
3 tensor(0.6133, grad_fn=<AddBackward0>)
4 tensor(0.5772, grad_fn=<AddBackward0>)
5 tensor(0.5511, grad_fn=<AddBackward0>)
6 tensor(0.5381, grad_fn=<AddBackward0>)
7 tensor(0.5349, grad_fn=<AddBackward0>)
8 tensor(0.5378, grad_fn=<AddBackward0>)
9 tensor(0.5402, grad_fn=<AddBackward0>)
10 tensor(0.5384, grad_fn=<AddBackward0>)
11 tensor(0.5357, grad_fn=<AddBackward0>)
12 tensor(0.5310, grad_fn=<AddBackward0>)
13 tensor(0.5256, grad_fn=<AddBackward0>)
14 tensor(0.5218, grad_fn=<AddBackward0>)
15 tensor(0.5183, grad_fn=<AddBackward0>)
16 tensor(0.5156, grad_fn=<AddBackward0>)
17 tensor(0.5139, grad_fn=<AddBackward0>)
18 tensor(0.5132, grad_fn=<AddBackward0>)
19 tensor(0.5122, grad_fn=<AddBackward0>)
20 tensor(0.5119, grad_fn=<AddBackward0>)
21 tensor(0.5114, grad_fn=<AddBackward0>)
22 tensor(0.5108, grad_fn=<AddBackward0>)
23 tensor(0.5097, grad_fn=<AddBackward0>)
24

2. Compute the mean predictions for 20 MC sampled models

In [43]:
mc_samples = 20
n_test_examples = y_test_np.shape[0]
y_test_mus_z = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    results = model(x_test_t_z)
    y_test_mus_z[i] = results[0].detach().numpy() # Get the predicted means for one sampled parameter vector

3. Compute the Mean Squared Error (MSE) for the variational distribution for the non-linear heteroscedastic regression model for the test data using 20 MC samples and Bayesian model averaging.

In [44]:
y_test_mu_z = np.mean(y_test_mus_z, axis=0) # Compute the mean predictions using Bayesian model averaging
y_test_mu = unscale(y_test_mu_z,y_mu,y_sigma)
print('MSE:', mean_squared_error(y_test_np, y_test_mu))

MSE: 5.67506961290825e-05


4. Compute the epistemic uncertainties for each regression test example.

In [45]:
y_test_mus = unscale(y_test_mus_z,y_mu,y_sigma)
y_test_epistemic = np.mean([(x - y_test_mu)**2 for x in y_test_mus], axis=0) # Compute the epistemic uncertainties

In [46]:
y_test_epistemic

array([[1.56418173e-08],
       [1.44423647e-08],
       [6.34635871e-09],
       ...,
       [1.66014657e-08],
       [2.04151653e-08],
       [2.66761322e-08]], shape=(2984, 1))

5. Select the best test example to add to the training set using epistemic-uncertainity-based active learning

In [47]:
selected_x_test = np.argmax(y_test_epistemic) # Get the index of the best test example to select for active learning

In [48]:
selected_x_test

np.int64(2714)

# Classification for Ship Detection


Load Ship Detection Dataset

In [49]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torch.utils.data import random_split
from torchvision.transforms.functional import resize
from sklearn import preprocessing
import numpy as np
from pathlib import Path
import torchmetrics

ROOT_PATH = "shipsnet"
LR = 1e-4
IMG_SIZE = [80]

tensor_size = IMG_SIZE[0]**2 * 3

def max_scaling(image):
    image = image / 255.0
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    return image

def normalize_img(image):
    means = torch.Tensor([[[105.0385]],[[108.1886]],[[ 94.9558]]])
    stds = torch.Tensor([[[48.4294]],[[40.0104]],[[38.6445]]])
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    image = image - means
    image = image / stds
    return image

#https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
class ShipDataset(Dataset):
    def __init__(self, root_path, transform = None):
        self.root_path = Path(root_path)
        self.files = list(self.root_path.rglob("*/*"))
        self.classes = list(set([int(entry.parts[-1]) for entry in self.root_path.rglob("*") if Path(entry).is_dir()]))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = read_image(str(self.files[idx]))
        label = int(self.files[idx].parts[-2])
        if self.transform:
            image = self.transform(image)
        return image, label


full_dataset  = ShipDataset(ROOT_PATH, transform = normalize_img,)
n_classes = len(full_dataset.classes)
n_examples = len(full_dataset)
train_dataset, test_dataset = random_split(full_dataset, [int(0.8*float(n_examples)), int(0.2*float(n_examples))])
n_train_examples = len(train_dataset)
train_dataloader = DataLoader(train_dataset, batch_size=len(train_dataset))
test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset))

criterion = torch.nn.BCELoss(reduce='mean')
accuracy = torchmetrics.classification.BinaryAccuracy()


C:\Users\shai1\PyCharmProjects\JupyterProject\Bayesian-lab-5\.venv\Lib\site-packages\torch\nn\modules\loss.py:48: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


In [50]:
n_train_examples = len(train_dataset)

6. Using PyTorch, perform variational inference using Concrete Dropout for a non-linear Bernoulli prediction model for the binary classification dataset

In [51]:
class ConcreteDropout(torch.nn.Module):

    """Concrete Dropout.

    Implementation of the Concrete Dropout module as described in the
    'Concrete Dropout' paper: https://arxiv.org/pdf/1705.07832
    """

    def __init__(self,
                 weight_regulariser: float,
                 dropout_regulariser: float,
                 init_min: float = 0.1,
                 init_max: float = 0.1) -> None:

        """Concrete Dropout.

        Parameters
        ----------
        weight_regulariser : float
            Weight regulariser term.
        dropout_regulariser : float
            Dropout regulariser term.
        init_min : float
            Initial min value.
        init_max : float
            Initial max value.
        """

        super().__init__()

        self.weight_regulariser = weight_regulariser
        self.dropout_regulariser = dropout_regulariser

        init_min = np.log(init_min) - np.log(1.0 - init_min)
        init_max = np.log(init_max) - np.log(1.0 - init_max)

        self.p_logit = torch.nn.parameter.Parameter(torch.empty(1).uniform_(init_min, init_max))
        self.p = torch.sigmoid(self.p_logit)

        self.regularisation = 0.0

    def forward(self, x: torch.Tensor, layer: torch.nn.Module) -> torch.Tensor:

        """Calculates the forward pass.

        The regularisation term for the layer is calculated and assigned to a
        class attribute - this can later be accessed to evaluate the loss.

        Parameters
        ----------
        x : Tensor
            Input to the Concrete Dropout.
        layer : nn.Module
            Layer for which to calculate the Concrete Dropout.

        Returns
        -------
        Tensor
            Output from the dropout layer.
        """

        output = layer(self._concrete_dropout(x))

        sum_of_squares = 0
        for param in layer.parameters():
            sum_of_squares += torch.sum(torch.pow(param, 2))

        weights_reg = self.weight_regulariser * sum_of_squares / (1.0 - self.p)

        dropout_reg = self.p * torch.log(self.p)
        dropout_reg += (1.0 - self.p) * torch.log(1.0 - self.p)
        dropout_reg *= self.dropout_regulariser * x[0].numel()

        self.regularisation = weights_reg + dropout_reg

        return output

    def _concrete_dropout(self, x: torch.Tensor) -> torch.Tensor:

        """Computes the Concrete Dropout.

        Parameters
        ----------
        x : Tensor
            Input tensor to the Concrete Dropout layer.

        Returns
        -------
        Tensor
            Outputs from Concrete Dropout.
        """

        eps = 1e-7
        tmp = 0.1

        self.p = torch.sigmoid(self.p_logit)
        u_noise = torch.rand_like(x)

        drop_prob = (torch.log(self.p + eps) -
                     torch.log(1 - self.p + eps) +
                     torch.log(u_noise + eps) -
                     torch.log(1 - u_noise + eps))

        drop_prob = torch.sigmoid(drop_prob / tmp)

        random_tensor = 1 - drop_prob
        retain_prob = 1 - self.p

        x = torch.mul(x, random_tensor) / retain_prob

        return x

In [52]:
w = 1./(100.*float(n_train_examples))
d = 1./float(n_train_examples)
class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = torch.nn.Linear(inputSize, hiddenSize)
        self.layer2 = torch.nn.Linear(hiddenSize, hiddenSize)
        self.layer3 = torch.nn.Linear(hiddenSize, outputSize)
        self.cd1 = ConcreteDropout(1e-6, 1e-6) # Call the constructor for ConcreteDropout
        self.cd2 = ConcreteDropout(1e-6, 1e-6) # Call the constructor for ConcreteDropout
        self.cd3 = ConcreteDropout(1e-6, 1e-6) # Call the constructor for ConcreteDropout
        self.relu = torch.nn.ReLU()
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        h1 = self.cd1(x, torch.nn.Sequential(self.layer1,self.relu))
        h2 = self.cd2(h1, torch.nn.Sequential(self.layer2,self.relu))
        l = self.cd3(h2, self.layer3)
        return torch.nn.functional.sigmoid(l)

    def reg(self):
        reg = 0.0
        reg += self.cd1.regularisation
        reg += self.cd2.regularisation
        reg += self.cd3.regularisation
        return reg

In [53]:
model = nn(tensor_size, 200, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(50):
    for data, label in train_dataloader:
        optimizer.zero_grad()
        p = model(data)
        nll_loss = criterion(p.squeeze(),label.float().squeeze())
        loss = nll_loss + model.reg() # Get variational loss for the model
        acc = accuracy(p.squeeze(), label.float().squeeze())
        loss.backward()
        optimizer.step()
        if verbose_option: print(epoch, loss, acc)

0 tensor([0.6671], grad_fn=<AddBackward0>) tensor(0.7122)
1 tensor([0.5711], grad_fn=<AddBackward0>) tensor(0.7806)
2 tensor([0.5293], grad_fn=<AddBackward0>) tensor(0.7869)
3 tensor([0.4966], grad_fn=<AddBackward0>) tensor(0.8109)
4 tensor([0.4696], grad_fn=<AddBackward0>) tensor(0.8216)
5 tensor([0.4408], grad_fn=<AddBackward0>) tensor(0.8250)
6 tensor([0.4217], grad_fn=<AddBackward0>) tensor(0.8328)
7 tensor([0.4039], grad_fn=<AddBackward0>) tensor(0.8413)
8 tensor([0.3931], grad_fn=<AddBackward0>) tensor(0.8453)
9 tensor([0.3782], grad_fn=<AddBackward0>) tensor(0.8531)
10 tensor([0.3611], grad_fn=<AddBackward0>) tensor(0.8653)
11 tensor([0.3433], grad_fn=<AddBackward0>) tensor(0.8719)
12 tensor([0.3324], grad_fn=<AddBackward0>) tensor(0.8784)
13 tensor([0.3171], grad_fn=<AddBackward0>) tensor(0.8847)
14 tensor([0.3055], grad_fn=<AddBackward0>) tensor(0.8916)
15 tensor([0.2948], grad_fn=<AddBackward0>) tensor(0.8928)
16 tensor([0.2795], grad_fn=<AddBackward0>) tensor(0.9000)
17 tens

7.  Compute the predicted probabilities and entropy predictions for 20 MC sampled models

In [54]:
mc_samples = 20
n_test_examples = len(test_dataset)
y_test_probs = np.zeros([mc_samples,n_test_examples,1])
y_test_entropies = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    for data, label in test_dataloader:
        results = model(data).detach().numpy()
        y_test_probs[i] = results # Get the predicted means for one sampled parameter vector
        y_test_entropies[i] += -(results * np.log(results) + (1-results) * np.log(1-results)) # Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector

8. Compute the Bayesian model averaging predictions for each classification test example.

In [55]:
y_test_probs_avg = np.mean(y_test_probs, axis=0) # Compute Bayesian model averaging predictions

In [56]:
y_test_probs_avg

array([[2.39388927e-03],
       [2.43601488e-03],
       [2.25771333e-03],
       [9.45966840e-01],
       [2.95836066e-01],
       [9.89604861e-01],
       [1.88620383e-02],
       [9.52311584e-01],
       [9.44407830e-01],
       [9.18275994e-01],
       [9.76466659e-01],
       [4.41722387e-03],
       [3.19625220e-04],
       [4.48113928e-02],
       [6.21183988e-01],
       [2.28451559e-01],
       [2.36461693e-05],
       [3.02217000e-02],
       [1.93786741e-01],
       [4.87345656e-05],
       [3.31692336e-04],
       [1.21009768e-01],
       [1.48256381e-03],
       [9.58135986e-01],
       [8.73453310e-07],
       [2.89000377e-01],
       [2.39361330e-02],
       [7.44152850e-01],
       [4.84814159e-07],
       [5.51591828e-04],
       [8.74950182e-01],
       [2.39354520e-03],
       [2.37013040e-01],
       [2.46248589e-04],
       [1.13470351e-01],
       [5.98053899e-01],
       [5.20226126e-05],
       [2.39821430e-01],
       [1.08013089e-02],
       [9.48131716e-01],


9. Compute the aleatoric uncertainty for each classification test example.

In [57]:
y_test_aleatoric  = np.mean(y_test_entropies, axis=0) # Computer aleatoric uncertainty

In [58]:
y_test_aleatoric

array([[1.55318745e-02],
       [1.68594963e-02],
       [1.53348965e-02],
       [2.08535381e-01],
       [6.02107435e-01],
       [5.63593687e-02],
       [9.26407684e-02],
       [1.78843270e-01],
       [2.12807494e-01],
       [2.70205341e-01],
       [1.08401377e-01],
       [2.66482909e-02],
       [2.73073226e-03],
       [1.71318788e-01],
       [6.56951082e-01],
       [5.28812133e-01],
       [2.60707629e-04],
       [1.33402656e-01],
       [4.74893172e-01],
       [5.09475975e-04],
       [2.95165394e-03],
       [3.67735867e-01],
       [1.08765044e-02],
       [1.69042623e-01],
       [1.27867437e-05],
       [5.77197091e-01],
       [1.12225037e-01],
       [5.60787992e-01],
       [7.33938989e-06],
       [4.56363602e-03],
       [3.74381511e-01],
       [1.64109963e-02],
       [5.40733612e-01],
       [2.24494164e-03],
       [3.53300723e-01],
       [6.69789234e-01],
       [5.46492000e-04],
       [5.40522675e-01],
       [5.70099401e-02],
       [2.02271705e-01],


10. Compute the epistemic uncertainty for each classification test example.

In [59]:
y_test_uncertainty = y_test_probs_avg * -1.*np.log(y_test_probs_avg) + (1. - y_test_probs_avg) * -1.*np.log(1. -y_test_probs_avg) # Compute the total uncertainity
y_test_epistemic = y_test_uncertainty - y_test_aleatoric # Compute the epistemic uncertainty

In [60]:
y_test_epistemic

array([[1.30587608e-03],
       [2.32005053e-04],
       [6.77422897e-04],
       [1.68822498e-03],
       [5.18738204e-03],
       [1.45009398e-03],
       [9.35935992e-04],
       [1.28085671e-02],
       [1.85538629e-03],
       [1.27545915e-02],
       [3.08728656e-03],
       [1.71042763e-03],
       [1.61301172e-04],
       [1.16257299e-02],
       [6.53044915e-03],
       [8.58643010e-03],
       [1.48245731e-05],
       [2.10932079e-03],
       [1.67742538e-02],
       [2.31488513e-05],
       [3.72710946e-05],
       [1.19619974e-03],
       [2.62354420e-04],
       [4.78087754e-03],
       [2.72091436e-07],
       [2.40571167e-02],
       [7.60725310e-04],
       [7.88019975e-03],
       [1.94379711e-07],
       [1.26232888e-04],
       [2.48557950e-03],
       [4.24678766e-04],
       [6.87878164e-03],
       [4.73977674e-05],
       [4.09268444e-04],
       [4.00362311e-03],
       [1.86715739e-05],
       [1.03513318e-02],
       [2.64210019e-03],
       [1.70826938e-03],


In [61]:
from sklearn.metrics import classification_report

for data, label in test_dataloader:
    print(classification_report(label.flatten(), y_test_probs_avg.round().flatten()))

              precision    recall  f1-score   support

           0       0.98      0.96      0.97       587
           1       0.90      0.94      0.92       213

    accuracy                           0.96       800
   macro avg       0.94      0.95      0.94       800
weighted avg       0.96      0.96      0.96       800

